# Clase 060 — Model Cards + Responsible ML

Implementamos un `ModelCard` dataclass al estilo Mitchell et al. (2019) y lo rellenamos automáticamente para un LogReg entrenado sobre datos sintéticos con un atributo protegido binario.

In [ ]:
import json
import numpy as np
import pandas as pd
from dataclasses import dataclass, field, asdict
from typing import Any
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

rng = np.random.default_rng(42)
np.random.seed(42)

## 1. Dataset sintético con atributo protegido

`sex` (0/1) es el atributo sensible. El target tiene un sesgo leve (mayor base rate en `sex=1`).

In [ ]:
n = 4000
sex = rng.integers(0, 2, n)
age = rng.normal(40, 12, n).clip(18, 80)
income = rng.normal(50_000, 15_000, n).clip(10_000, 200_000)
score = rng.normal(0, 1, n)
logit = 0.02 * (age - 40) + 0.00002 * (income - 50_000) + 0.5 * score + 0.4 * sex - 0.5
p = 1 / (1 + np.exp(-logit))
y = (rng.uniform(0, 1, n) < p).astype(int)

df = pd.DataFrame({'sex': sex, 'age': age, 'income': income, 'score': score})
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.3, random_state=42, stratify=y)
print('train', X_train.shape, 'test', X_test.shape, 'pos rate', y.mean().round(3))

## 2. Entrenar LogReg

In [ ]:
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(f'accuracy global: {accuracy_score(y_test, preds):.4f}')

## 3. Dataclass `ModelCard` (Mitchell et al. 2019)

In [ ]:
@dataclass
class ModelCard:
    model_name: str
    version: str
    intended_use: dict[str, Any] = field(default_factory=dict)
    factors: dict[str, Any] = field(default_factory=dict)
    metrics_per_subgroup: dict[str, Any] = field(default_factory=dict)
    training_data_summary: dict[str, Any] = field(default_factory=dict)
    evaluation_data_summary: dict[str, Any] = field(default_factory=dict)
    ethical_considerations: list[str] = field(default_factory=list)
    caveats_and_recommendations: list[str] = field(default_factory=list)

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2, default=str)

    def to_markdown(self) -> str:
        d = asdict(self)
        lines = [f"# Model Card — {d['model_name']} v{d['version']}\n"]
        for key in ['intended_use', 'factors', 'training_data_summary', 'evaluation_data_summary']:
            lines.append(f'## {key.replace("_", " ").title()}\n')
            for k, v in d[key].items():
                lines.append(f'- **{k}**: {v}')
            lines.append('')
        lines.append('## Metrics per Subgroup\n')
        for group, metrics in d['metrics_per_subgroup'].items():
            lines.append(f'### {group}')
            for k, v in metrics.items():
                lines.append(f'- {k}: {v:.4f}' if isinstance(v, float) else f'- {k}: {v}')
            lines.append('')
        lines.append('## Ethical Considerations\n')
        for item in d['ethical_considerations']:
            lines.append(f'- {item}')
        lines.append('\n## Caveats and Recommendations\n')
        for item in d['caveats_and_recommendations']:
            lines.append(f'- {item}')
        return '\n'.join(lines)

## 4. Métricas por subgrupo (accuracy, TPR, FPR)

In [ ]:
def subgroup_metrics(y_true, y_pred, sensitive):
    out = {}
    for g in np.unique(sensitive):
        mask = sensitive == g
        if mask.sum() == 0:
            continue
        tn, fp, fn, tp = confusion_matrix(y_true[mask], y_pred[mask], labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) else 0.0
        fpr = fp / (fp + tn) if (fp + tn) else 0.0
        out[f'sex={int(g)}'] = {
            'n': int(mask.sum()),
            'accuracy': float(accuracy_score(y_true[mask], y_pred[mask])),
            'TPR': float(tpr),
            'FPR': float(fpr),
        }
    return out

sens_test = X_test['sex'].values
sub_metrics = subgroup_metrics(y_test, preds, sens_test)
print(pd.DataFrame(sub_metrics).T.round(4))

## 5. `build_model_card(...)` autorrellena

In [ ]:
def build_model_card(model, X_train, y_train, sensitive_attr, X_test, y_test):
    preds = model.predict(X_test)
    card = ModelCard(
        model_name=type(model).__name__,
        version='1.0.0',
        intended_use={
            'primary_users': 'Equipo de análisis de riesgo',
            'primary_uses': 'Scoring de probabilidad de evento positivo (demo)',
            'out_of_scope': 'NO usar para decisiones de crédito o empleo en producción sin auditoría',
        },
        factors={
            'demographic': sensitive_attr,
            'environmental': 'datos sintéticos seed=42',
        },
        metrics_per_subgroup=subgroup_metrics(y_test, preds, X_test[sensitive_attr].values),
        training_data_summary={
            'n_train': int(len(X_train)),
            'pos_rate': float(np.mean(y_train).round(4)),
            'features': list(X_train.columns),
        },
        evaluation_data_summary={
            'n_test': int(len(X_test)),
            'pos_rate': float(np.mean(y_test).round(4)),
        },
        ethical_considerations=[
            f'Posible disparidad por {sensitive_attr} — ver metrics_per_subgroup',
            'Datos sintéticos: no representan población real',
            'No se aplicó mitigación de sesgo — baseline para comparar',
        ],
        caveats_and_recommendations=[
            'Calibración no verificada — reportar Brier antes de producción',
            'Distribution shift no monitoreado',
            'Re-evaluar trimestralmente',
        ],
    )
    return card

card = build_model_card(model, X_train, y_train, 'sex', X_test, y_test)

## 6. Render a Markdown y JSON

In [ ]:
print(card.to_markdown())

In [ ]:
print(card.to_json()[:1200])
print('... (truncado)')

## Ejercicios

1. Agregá una sección `quantitative_analyses` con calibración (Brier por subgrupo).
2. Implementá `to_html()` con tabla CSS mínima.
3. Integrá con `model-card-toolkit` de Google (opcional).

## Conclusiones

- Un Model Card es ante todo un contrato — propone qué usos sí y cuáles no.
- Reportar métricas por subgrupo expone disparidades que el global esconde.
- EU AI Act + NIST AI RMF empujan esto de buena práctica a requisito legal.

## ✅ Soluciones de los ejercicios

Reutilizamos el `ModelCard`, `subgroup_metrics`, el `model` (LogReg) y el split ya definidos arriba. Los datasets externos que menciona el README (California Housing, credit-g) requieren descarga; usamos el dataset sintético en memoria, que ya trae el atributo protegido `sex`.

**Ej. 1 — Model Card básico.** Construimos la card, la renderizamos a Markdown y la persistimos como `MODEL_CARD.md` (en un temporal). Verificamos que trae todas las secciones.

In [ ]:

import os, tempfile
card = build_model_card(model, X_train, y_train, "sex", X_test, y_test)
md = card.to_markdown()
secciones = ["Intended Use", "Factors", "Metrics per Subgroup",
             "Ethical Considerations", "Caveats and Recommendations"]
for s in secciones:
    assert s in md, f"falta seccion {s}"
p = os.path.join(tempfile.gettempdir(), "MODEL_CARD.md")
open(p, "w", encoding="utf-8").write(md)
assert os.path.getsize(p) > 200
print(f"MODEL_CARD.md escrita ({os.path.getsize(p)} bytes) con {len(secciones)}+ secciones")
print(md[:400], "...")
os.remove(p)

**Ej. 2 — Subgroup metrics.** Accuracy y FPR desagregadas por `sex` y por `age_group`. El global puede ocultar disparidades entre subgrupos.

In [ ]:

import numpy as np, pandas as pd

preds_te = model.predict(X_test)
por_sex = subgroup_metrics(y_test, preds_te, X_test["sex"].values)
age_group = pd.cut(X_test["age"], bins=[0, 30, 45, 60, 120], labels=[0, 1, 2, 3]).astype(int).values
por_age = subgroup_metrics(y_test, preds_te, age_group)

print("=== por sex ===");      print(pd.DataFrame(por_sex).T[["accuracy", "FPR"]].round(4))
print("=== por age_group ==="); print(pd.DataFrame(por_age).T[["accuracy", "FPR"]].round(4))
fpr_gap = abs(por_sex["sex=0"]["FPR"] - por_sex["sex=1"]["FPR"])
print(f"\nbrecha de FPR entre sexos: {fpr_gap:.4f} (una disparidad a vigilar)")
assert len(por_sex) == 2 and len(por_age) >= 2

**Ej. 3 — Clasificación de riesgo (EU AI Act).** Cada caso de uso a su tier: `unacceptable` / `high` / `limited` / `minimal`.

In [ ]:

eu_ai_act = {
 "recomendacion_peliculas": "minimal",   # sin impacto sobre derechos
 "score_crediticio":        "high",       # acceso a servicios esenciales
 "seleccion_rrhh":          "high",       # empleo: alto riesgo
 "marketing_email":         "minimal",    # riesgo bajo
 "detector_spam":           "minimal",    # riesgo bajo
}
for caso, tier in eu_ai_act.items():
    print(f"{caso:26s} -> {tier}")
assert eu_ai_act["score_crediticio"] == "high" and eu_ai_act["seleccion_rrhh"] == "high"
print("regla: si afecta acceso a credito, empleo, educacion o justicia -> high-risk")

**Ej. 4 — HuggingFace Card (template).** Generamos el frontmatter YAML + cuerpo al estilo Hub. No subimos nada (sin conexión y para no publicar sin querer); solo validamos la estructura.

In [ ]:

hf_card = """---
license: mit
tags: [tabular-classification, responsible-ai]
metrics: [accuracy]
model-index:
- name: logreg-demo
  results:
  - task: {type: tabular-classification}
    metrics:
    - {type: accuracy, value: %.4f}
---

# logreg-demo
Modelo de demostracion. Uso previsto: scoring interno. Fuera de alcance: credito/empleo.
""" % model.score(X_test, y_test)
print(hf_card)
assert hf_card.startswith("---") and "model-index" in hf_card and "accuracy" in hf_card
print("OK: card estilo HuggingFace con frontmatter YAML valido (no publicada)")

**Ej. 5 — NIST AI RMF.** Las 4 funciones del framework: Map (contexto), Measure (métricas), Manage (mitigaciones), Govern (ownership).

In [ ]:

nist_rmf = {
 "Map":     "contexto: scoring de riesgo interno; stakeholders: analistas; atributo sensible sex",
 "Measure": "accuracy global + TPR/FPR por subgrupo; calibracion (Brier) pendiente",
 "Manage":  "mitigacion: umbral por subgrupo / reweighing; plan de rollback si crece la brecha",
 "Govern":  "owner: equipo de riesgo; revision trimestral; registro de versiones del modelo",
}
for func, contenido in nist_rmf.items():
    print(f"[{func}] {contenido}")
assert set(nist_rmf) == {"Map", "Measure", "Manage", "Govern"}
print("\nOK: las 4 funciones del NIST AI RMF cubiertas")